# MAC Multitask Curriculum Dataset — Universo 3 AS-OF para modelos locales y Financial-GFM

Checkpoint 19 separa el universo legacy local del dataset global causal.


## 0. Instalación opcional

Ejecuta sólo si el kernel Python 3 no tiene las librerías necesarias. En ADA, si dependes de Artifactory, instala con el mecanismo autorizado de tu entorno.

In [ ]:
# %pip install polars pyarrow boto3 awswrangler s3fs matplotlib pandas

## 1. Imports

In [ ]:
from __future__ import annotations

import io
import json
import math
import os
import time
from datetime import datetime
from urllib.parse import urlparse

import boto3
import matplotlib.pyplot as plt
import polars as pl

from global_long_schema import build_and_validate_global_long

try:
    import awswrangler as wr
except ImportError:
    wr = None

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(60)

## 2. Parámetros del experimento y detección AWS/Athena

El `workgroup` se detecta automáticamente. En tu ambiente debería elegir `sandbox`.

In [ ]:
# ------------------------------
# Tablas origen
# ------------------------------
# La query corre contra 2_tab_bo porque ahí vive deta_nivel1_base.
# El histórico MAC se referencia completamente como mx_master.t_mmfi_mac_historical_balances.
DATABASE = '2_tab_bo'
MAC_DATABASE = 'mx_master'
MAC_SOURCE_TABLE = 't_mmfi_mac_historical_balances'
MAC_FULL_TABLE = f'{MAC_DATABASE}.{MAC_SOURCE_TABLE}'

DETA_NIVEL1_TABLE = 'deta_nivel1_base'

# ------------------------------
# Parámetros temporales
# ------------------------------
MIN_CUTOFF_DATE = '2024-06-11'
MAX_CUTOFF_DATE = '2026-05-31'
FECHA_ANALISIS = '2026-04-30'

# ------------------------------
# Universo 3
# ------------------------------
# Sólo grupos entrenables y sólo CUADRE/DESCUADRE.
UNIVERSO_3_GROUPS = [
    'Grupo_2',  # positiva / cuentas_entrenamiento
    'Grupo_3',  # negativa / cuentas_entrenamiento
    'Grupo_5',  # ambos_signos / cuentas_entrenamiento
]

NIVEL_1_TYPES = [
    'CUADRE',
    'DESCUADRE',
]

# ------------------------------
# Currículum y selección LEGACY (sólo code_02 locales)
# ------------------------------
N_CURRICULUM_BUCKETS = 20
N_PER_GROUP = 10
N_BINS_ENTROPY = 10

# Este umbral se conserva exclusivamente para los datasets locales legacy.
# Financial-GFM usa TODO Universo 3 elegible y recalcula dificultad después del split,
# sólo con train y por cross_key_id final (saldo/variación separados).
MODEL_SCORE_THRESHOLD = 0.5

# Universo 3 debe correr completo por defecto.
# True  -> muestrea N_PER_GROUP por grupo/nivel_1 sólo para smoke rápido.
# False -> usa TODAS las cuenta-divisa de Universo 3.
# Para este análisis debe quedarse en False.
SAMPLE_MODE = False

# En Universo 3 el wide es manejable, pero si crece demasiado puedes apagarlo.
WRITE_WIDE_WHEN_ALL = True

# ------------------------------
# STS / región
# ------------------------------
sts = boto3.client('sts')
identity = sts.get_caller_identity()

AWS_ACCOUNT_ID = identity['Account']
AWS_ARN = identity['Arn']
AWS_USER_ID = identity['UserId']

AWS_REGION = (
    os.environ.get('AWS_REGION')
    or os.environ.get('AWS_DEFAULT_REGION')
    or boto3.session.Session().region_name
    or 'us-east-1'
)

athena = boto3.client('athena', region_name=AWS_REGION)
s3 = boto3.client('s3', region_name=AWS_REGION)

# ------------------------------
# Workgroup Athena
# ------------------------------
workgroups = athena.list_work_groups()['WorkGroups']
workgroup_names = [wg['Name'] for wg in workgroups]

preferred_workgroup = 'sandbox'
if preferred_workgroup in workgroup_names:
    ATHENA_WORKGROUP = preferred_workgroup
elif 'primary' in workgroup_names:
    ATHENA_WORKGROUP = 'primary'
else:
    ATHENA_WORKGROUP = workgroup_names[0]

workgroup_cfg = athena.get_work_group(WorkGroup=ATHENA_WORKGROUP)['WorkGroup']['Configuration']
ATHENA_QUERY_OUTPUT = workgroup_cfg.get('ResultConfiguration', {}).get('OutputLocation')
if ATHENA_QUERY_OUTPUT is None:
    ATHENA_QUERY_OUTPUT = 's3://your-private-bucket/users/your-user/athena_query_results/'

# ------------------------------
# Rutas S3
# ------------------------------
PATH_SERIES = (
    's3://your-private-bucket/'
    'users/your-user/mac_multitask_curriculum_universo3_score05/'
)

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
TODAY = datetime.now().strftime('%Y%m%d')

# Athena UNLOAD necesita escribir en una carpeta vacía.
PATH_BASE = f'{PATH_SERIES}base_canonica/run_id={RUN_ID}/'

# Outputs técnicos del experimento.
PATH_PARQUET = f'{PATH_SERIES}parquet/run_id={RUN_ID}/'
PATH_CSV = f'{PATH_SERIES}csv/run_id={RUN_ID}/'
PATH_FIGURES = f'{PATH_SERIES}figures/run_id={RUN_ID}/'

# Salida compatible con code_02_MLP_E_D.ipynb:
# code_02 detecta carpetas tipo YYYYMMDD_YYYYMMDD_YYYYMMDD bajo este prefijo.
BUCKET_COMPAT = 'your-private-bucket'
PREFIX_COMPAT_BASE = 'data/sandboxes/m6hn/data/coe_liquidez_diaria/csv/'
FOLDER_ID = f'{TODAY}_{TODAY}_{TODAY}'
PATH_COMPAT_FOLDER = f's3://{BUCKET_COMPAT}/{PREFIX_COMPAT_BASE}{FOLDER_ID}/'
PATH_FULL_DIFF_CSV = f'{PATH_COMPAT_FOLDER}full_diff.csv'
PATH_FULL_DIFF_ROBUST_CSV = f'{PATH_COMPAT_FOLDER}full_diff_robust.csv'
PATH_SERIES_LONG_CSV = f'{PATH_COMPAT_FOLDER}series_long.csv'
PATH_SERIES_LONG_ROBUST_CSV = f'{PATH_COMPAT_FOLDER}series_long_robust.csv'
PATH_GLOBAL_SERIES_LONG_CSV = f'{PATH_COMPAT_FOLDER}global_series_long.csv'
PATH_GLOBAL_SERIES_LONG_PARQUET = f'{PATH_COMPAT_FOLDER}global_series_long.parquet'
PATH_GLOBAL_LONG_VALIDATION_JSON = f'{PATH_COMPAT_FOLDER}global_series_long_validation.json'
PATH_SELECTED_CURRICULUM_CSV = f'{PATH_COMPAT_FOLDER}selected_curriculum_table.csv'
PATH_CURRICULUM_STATS_CSV = f'{PATH_COMPAT_FOLDER}curriculum_stats_all_series.csv'

print('AWS_ACCOUNT_ID:', AWS_ACCOUNT_ID)
print('AWS_ARN:', AWS_ARN)
print('AWS_USER_ID:', AWS_USER_ID)
print('AWS_REGION:', AWS_REGION)
print('Athena workgroups disponibles:', workgroup_names)
print('ATHENA_WORKGROUP:', ATHENA_WORKGROUP)
print('ATHENA_QUERY_OUTPUT:', ATHENA_QUERY_OUTPUT)
print('RUN_ID:', RUN_ID)
print('MAC_FULL_TABLE:', MAC_FULL_TABLE)
print('DETA_NIVEL1_TABLE:', DETA_NIVEL1_TABLE)
print('FECHA_ANALISIS:', FECHA_ANALISIS)
print('UNIVERSO_3_GROUPS:', UNIVERSO_3_GROUPS)
print('NIVEL_1_TYPES:', NIVEL_1_TYPES)
print('MODEL_SCORE_THRESHOLD:', MODEL_SCORE_THRESHOLD)
print('PATH_BASE:', PATH_BASE)
print('PATH_COMPAT_FOLDER:', PATH_COMPAT_FOLDER)
print('PATH_FULL_DIFF_CSV:', PATH_FULL_DIFF_CSV)
print('PATH_GLOBAL_SERIES_LONG_PARQUET:', PATH_GLOBAL_SERIES_LONG_PARQUET)


## 3. Helpers S3 y Athena

Nota: Athena `UNLOAD` en Parquet puede generar archivos **sin extensión `.parquet`**, por eso la lectura lista archivos de datos bajo el prefijo y no filtra por sufijo.

In [ ]:
def parse_s3_uri(uri: str) -> tuple[str, str]:
    parsed = urlparse(uri)
    if parsed.scheme != 's3':
        raise ValueError(f'URI inválida: {uri}')
    return parsed.netloc, parsed.path.lstrip('/')


def list_s3_objects(uri: str) -> list[str]:
    bucket, prefix = parse_s3_uri(uri)
    paginator = s3.get_paginator('list_objects_v2')
    objects = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            objects.append(obj['Key'])
    return objects


def assert_s3_prefix_empty(uri: str) -> None:
    objects = list_s3_objects(uri)
    if objects:
        raise ValueError(
            f'El prefijo S3 no está vacío: {uri}\n'
            f'Athena UNLOAD requiere una carpeta vacía. Objetos encontrados: {len(objects)}'
        )


def list_s3_data_files(uri: str) -> list[str]:
    bucket, prefix = parse_s3_uri(uri)
    paginator = s3.get_paginator('list_objects_v2')
    files = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            name = key.split('/')[-1]
            if not name:
                continue
            if name.endswith('.metadata') or name.endswith('.manifest') or 'manifest' in name.lower():
                continue
            files.append(f's3://{bucket}/{key}')
    return files


def run_athena_query(
    query: str,
    database: str | None = DATABASE,
    workgroup: str = ATHENA_WORKGROUP,
    output_location: str = ATHENA_QUERY_OUTPUT,
    poll_seconds: int = 5,
) -> str:
    params = {
        'QueryString': query,
        'WorkGroup': workgroup,
        'ResultConfiguration': {'OutputLocation': output_location},
    }
    if database:
        params['QueryExecutionContext'] = {'Database': database}

    response = athena.start_query_execution(**params)
    query_id = response['QueryExecutionId']
    print('Athena query_id:', query_id)

    while True:
        execution = athena.get_query_execution(QueryExecutionId=query_id)['QueryExecution']
        state = execution['Status']['State']
        if state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
            break
        time.sleep(poll_seconds)

    if state != 'SUCCEEDED':
        reason = execution['Status'].get('StateChangeReason', '')
        raise RuntimeError(f'Athena terminó en {state}. Razón: {reason}. QueryId: {query_id}')

    print('Athena query status:', state)
    return query_id


def read_athena_unload_parquet_polars(uri: str) -> pl.DataFrame:
    data_files = list_s3_data_files(uri)
    if not data_files:
        raise FileNotFoundError(f'No se encontraron archivos de datos en {uri}')

    print('Archivos detectados:', len(data_files))
    print('Primer archivo:', data_files[0])

    try:
        return pl.read_parquet(data_files, storage_options={'region': AWS_REGION})
    except Exception as exc:
        print('Polars no pudo leer directo desde S3. Intentando fallback con awswrangler...')
        print(type(exc).__name__, str(exc)[:500])
        if wr is None:
            raise ImportError('awswrangler no está instalado y Polars no pudo leer S3 directo.') from exc
        pdf = wr.s3.read_parquet(path=data_files, boto3_session=boto3.Session(region_name=AWS_REGION))
        return pl.from_pandas(pdf)


def write_text_to_s3(text: str, uri: str, content_type: str = 'text/plain') -> None:
    bucket, key = parse_s3_uri(uri)
    s3.put_object(Bucket=bucket, Key=key, Body=text.encode('utf-8'), ContentType=content_type)
    print(f'Archivo escrito: {uri}')


def write_polars_csv_to_s3(df: pl.DataFrame, uri: str) -> None:
    buffer = io.StringIO()
    df.write_csv(buffer)
    write_text_to_s3(buffer.getvalue(), uri, content_type='text/csv')


def write_polars_parquet_to_s3(df: pl.DataFrame, uri: str) -> None:
    bucket, key = parse_s3_uri(uri)
    buffer = io.BytesIO()
    df.write_parquet(buffer)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    print(f'Archivo escrito: {uri}')

## 4. Athena: Universo 3 AS-OF en Parquet

Athena hace la parte pesada:

1. Construye **Universo 2 Nivel 1** desde `deta_nivel1_base`:
   - `key_cols = clave_interfaz, cuenta, centro, divisa_raw`
   - `lag` por `fecha ASC`
   - `diferencia_estanco = saldo_aplicativo_estanco - saldo_contable_estanco`
   - marca `CUADRE` / `DESCUADRE`.
2. Construye **Universo 1 AS-OF** desde el histórico MAC:
   - toma el último estado conocido de cada `cross_key_id` con `cutoff_date <= FECHA_ANALISIS`.
   - evita mezclar estados viejos como `cuentas_cero` o `cuentas_atipicos` cuando el estado vigente ya es `cuentas_entrenamiento`.
3. Cruza por:

```sql
cross_key_id = cuenta || divisa_normalizada
```

4. Se queda sólo con:
   - `Grupo_2 = positiva / cuentas_entrenamiento`
   - `Grupo_3 = negativa / cuentas_entrenamiento`
   - `Grupo_5 = ambos_signos / cuentas_entrenamiento`
   - `nivel_1 IN ('CUADRE', 'DESCUADRE')`

La historia completa se trae desde `mx_master.t_mmfi_mac_historical_balances`, pero las etiquetas usadas por el modelo son las etiquetas AS-OF congeladas del universo, no las etiquetas crudas históricas fila a fila.


In [ ]:
grupo_sql = ', '.join([f"'{x}'" for x in UNIVERSO_3_GROUPS])
nivel_1_sql = ', '.join([f"'{x}'" for x in NIVEL_1_TYPES])

query_base = f"""
UNLOAD (
    WITH
    params AS (
        SELECT DATE '{FECHA_ANALISIS}' AS fecha_analisis
    ),

    grupo_catalogo AS (
        SELECT * FROM (
            VALUES
                ('Grupo_2', 'positiva / cuentas_entrenamiento', 'positiva', 'cuentas_entrenamiento', 2),
                ('Grupo_3', 'negativa / cuentas_entrenamiento', 'negativa', 'cuentas_entrenamiento', 3),
                ('Grupo_5', 'ambos_signos / cuentas_entrenamiento', 'ambos_signos', 'cuentas_entrenamiento', 5)
        ) AS t(
            grupo,
            nombre_grupo,
            classification_account_type,
            gl_account_hist_behaviour_type,
            orden
        )
        WHERE grupo IN ({grupo_sql})
    ),

    -- Estado vigente de cada cross_key_id al FECHA_ANALISIS.
    -- Esto evita que una misma cuenta-divisa contamine el dataset con estados viejos
    -- como cuentas_cero o cuentas_atipicos si actualmente es cuentas_entrenamiento.
    universo_1_asof AS (
        SELECT
            cross_key_id,
            classification_account_type,
            gl_account_hist_behaviour_type,
            label_cutoff_date
        FROM (
            SELECT
                TRIM(CAST(cross_key_id AS VARCHAR)) AS cross_key_id,
                classification_account_type,
                gl_account_hist_behaviour_type,
                CAST(cutoff_date AS DATE) AS label_cutoff_date,
                ROW_NUMBER() OVER (
                    PARTITION BY TRIM(CAST(cross_key_id AS VARCHAR))
                    ORDER BY CAST(cutoff_date AS DATE) DESC, load_date DESC
                ) AS rn
            FROM {MAC_FULL_TABLE}
            WHERE CAST(cutoff_date AS DATE) <= DATE '{FECHA_ANALISIS}'
        )
        WHERE rn = 1
    ),

    universo_1_entrenables AS (
        SELECT
            u.cross_key_id,
            g.grupo,
            g.nombre_grupo,
            g.orden,
            u.classification_account_type,
            u.gl_account_hist_behaviour_type,
            u.label_cutoff_date
        FROM universo_1_asof u
        INNER JOIN grupo_catalogo g
            ON u.classification_account_type = g.classification_account_type
           AND u.gl_account_hist_behaviour_type = g.gl_account_hist_behaviour_type
    ),

    deta_base AS (
        SELECT
            TRIM(CAST(clave_interfaz AS VARCHAR)) AS clave_interfaz,
            TRIM(CAST(cuenta AS VARCHAR)) AS cuenta,
            TRIM(CAST(centro AS VARCHAR)) AS centro,
            CAST(divisa_raw AS VARCHAR) AS divisa_raw,
            TRY_CAST(fecha AS DATE) AS fecha,
            CAST(saldo_aplicativo AS DECIMAL(28, 2)) AS saldo_aplicativo,
            CAST(saldo_contable AS DECIMAL(28, 2)) AS saldo_contable
        FROM {DETA_NIVEL1_TABLE}
        WHERE TRY_CAST(fecha AS DATE) IS NOT NULL
    ),

    deta_lag AS (
        SELECT
            d.*,

            LAG(fecha, 1) OVER (
                PARTITION BY clave_interfaz, cuenta, centro, divisa_raw
                ORDER BY fecha ASC
            ) AS fecha_anterior,

            LAG(saldo_aplicativo, 1) OVER (
                PARTITION BY clave_interfaz, cuenta, centro, divisa_raw
                ORDER BY fecha ASC
            ) AS saldo_aplicativo_anterior,

            LAG(saldo_contable, 1) OVER (
                PARTITION BY clave_interfaz, cuenta, centro, divisa_raw
                ORDER BY fecha ASC
            ) AS saldo_contable_anterior

        FROM deta_base d
    ),

    universo_2_key AS (
        SELECT
            clave_interfaz,
            cuenta,
            centro,

            CASE
                WHEN divisa_raw IS NULL OR TRIM(divisa_raw) = '' THEN 'MXP'
                ELSE TRIM(divisa_raw)
            END AS divisa,

            CONCAT(
                cuenta,
                CASE
                    WHEN divisa_raw IS NULL OR TRIM(divisa_raw) = '' THEN 'MXP'
                    ELSE TRIM(divisa_raw)
                END
            ) AS cross_key_id,

            fecha,
            fecha_anterior,

            CAST(
                saldo_aplicativo - saldo_aplicativo_anterior
                AS DECIMAL(28, 2)
            ) AS saldo_aplicativo_estanco,

            CAST(
                saldo_contable - saldo_contable_anterior
                AS DECIMAL(28, 2)
            ) AS saldo_contable_estanco,

            CAST(
                (saldo_aplicativo - saldo_aplicativo_anterior)
                -
                (saldo_contable - saldo_contable_anterior)
                AS DECIMAL(28, 2)
            ) AS diferencia_estanco,

            CASE
                WHEN fecha_anterior IS NULL THEN 'RESIDUO'

                WHEN CAST(
                    (saldo_aplicativo - saldo_aplicativo_anterior)
                    -
                    (saldo_contable - saldo_contable_anterior)
                    AS DECIMAL(28, 2)
                ) = CAST(0 AS DECIMAL(28, 2))
                    THEN 'CUADRE'

                WHEN CAST(
                    (saldo_aplicativo - saldo_aplicativo_anterior)
                    -
                    (saldo_contable - saldo_contable_anterior)
                    AS DECIMAL(28, 2)
                ) <> CAST(0 AS DECIMAL(28, 2))
                    THEN 'DESCUADRE'

                ELSE 'RESIDUO'
            END AS nivel_1

        FROM deta_lag
        WHERE fecha = (SELECT fecha_analisis FROM params)
    ),

    universo_2 AS (
        SELECT
            cross_key_id,
            MAX(cuenta) AS cuenta,
            MAX(divisa) AS divisa,

            CASE
                WHEN SUM(CASE WHEN nivel_1 = 'DESCUADRE' THEN 1 ELSE 0 END) > 0
                    THEN 'DESCUADRE'
                WHEN SUM(CASE WHEN nivel_1 = 'CUADRE' THEN 1 ELSE 0 END) > 0
                    THEN 'CUADRE'
                ELSE 'RESIDUO'
            END AS nivel_1,

            COUNT(*) AS n_llaves,
            COUNT(DISTINCT clave_interfaz) AS n_interfaces,
            COUNT(DISTINCT centro) AS n_centros,
            SUM(CASE WHEN nivel_1 = 'CUADRE' THEN 1 ELSE 0 END) AS n_llaves_cuadre,
            SUM(CASE WHEN nivel_1 = 'DESCUADRE' THEN 1 ELSE 0 END) AS n_llaves_descuadre,
            SUM(COALESCE(diferencia_estanco, CAST(0 AS DECIMAL(28, 2)))) AS total_diferencia_estanco
        FROM universo_2_key
        GROUP BY cross_key_id
    ),

    universo_3 AS (
        SELECT
            u1.cross_key_id,
            u1.grupo,
            u1.nombre_grupo,
            u1.orden,
            u1.classification_account_type,
            u1.gl_account_hist_behaviour_type,
            u1.label_cutoff_date,
            u2.nivel_1,
            u2.cuenta,
            u2.divisa,
            u2.n_llaves,
            u2.n_interfaces,
            u2.n_centros,
            u2.n_llaves_cuadre,
            u2.n_llaves_descuadre,
            u2.total_diferencia_estanco
        FROM universo_1_entrenables u1
        INNER JOIN universo_2 u2
            ON u1.cross_key_id = u2.cross_key_id
        WHERE u2.nivel_1 IN ({nivel_1_sql})
    ),

    ranked AS (
        SELECT
            TRIM(CAST(h.cross_key_id AS VARCHAR)) AS cross_key_id,
            CAST(h.cutoff_date AS DATE) AS cutoff_date,
            h.load_date,
            h.currency_id,

            -- Etiquetas congeladas del Universo 3 AS-OF.
            u3.classification_account_type AS classification_account_type,
            u3.gl_account_hist_behaviour_type AS gl_account_hist_behaviour_type,
            u3.label_cutoff_date AS universe_label_cutoff_date,

            -- Etiquetas crudas de la fila histórica, sólo para auditoría.
            h.classification_account_type AS raw_hist_classification_account_type,
            h.gl_account_hist_behaviour_type AS raw_hist_behaviour_type,

            u3.grupo,
            u3.nombre_grupo,
            u3.nivel_1,
            u3.cuenta,
            u3.divisa,
            u3.n_llaves,
            u3.n_interfaces,
            u3.n_centros,
            u3.n_llaves_cuadre,
            u3.n_llaves_descuadre,
            u3.total_diferencia_estanco,
            h.gl_mthly_ab_mxn_amount,
            h.prev_day_bal_cbl_vary_amount,
            ROW_NUMBER() OVER (
                PARTITION BY TRIM(CAST(h.cross_key_id AS VARCHAR)), CAST(h.cutoff_date AS DATE)
                ORDER BY h.load_date DESC
            ) AS rn
        FROM {MAC_FULL_TABLE} h
        INNER JOIN universo_3 u3
            ON TRIM(CAST(h.cross_key_id AS VARCHAR)) = u3.cross_key_id
        WHERE CAST(h.cutoff_date AS DATE) BETWEEN DATE '{MIN_CUTOFF_DATE}' AND DATE '{MAX_CUTOFF_DATE}'
    )

    SELECT
        cross_key_id,
        cutoff_date,
        currency_id,
        classification_account_type,
        gl_account_hist_behaviour_type,
        universe_label_cutoff_date,
        raw_hist_classification_account_type,
        raw_hist_behaviour_type,
        grupo,
        nombre_grupo,
        nivel_1,
        cuenta,
        divisa,
        n_llaves,
        n_interfaces,
        n_centros,
        n_llaves_cuadre,
        n_llaves_descuadre,
        CAST(total_diferencia_estanco AS DOUBLE) AS total_diferencia_estanco,
        CAST(gl_mthly_ab_mxn_amount AS DOUBLE) AS saldo,
        CAST(prev_day_bal_cbl_vary_amount AS DOUBLE) AS variacion
    FROM ranked
    WHERE rn = 1
)
TO '{PATH_BASE}'
WITH (
    format = 'PARQUET',
    compression = 'SNAPPY'
)
"""

print(query_base)


In [ ]:
# Ejecutar una sola vez por RUN_ID.
assert_s3_prefix_empty(PATH_BASE)
query_id_base = run_athena_query(query_base)
print('query_id_base:', query_id_base)
print('Objetos generados:', len(list_s3_data_files(PATH_BASE)))

## 5. Polars: lectura y validación de la base canónica

In [ ]:
base_raw = read_athena_unload_parquet_polars(PATH_BASE)
print(base_raw.shape)
base_raw.head(5)

In [ ]:
base_summary = (
    base_raw
    .group_by(['grupo', 'nombre_grupo', 'nivel_1', 'classification_account_type', 'gl_account_hist_behaviour_type'])
    .agg([
        pl.col('cross_key_id').n_unique().alias('cuenta_divisa'),
        pl.len().alias('observaciones'),
        pl.col('cutoff_date').min().alias('min_cutoff_date'),
        pl.col('cutoff_date').max().alias('max_cutoff_date'),
        pl.col('universe_label_cutoff_date').min().alias('min_universe_label_cutoff_date'),
        pl.col('universe_label_cutoff_date').max().alias('max_universe_label_cutoff_date'),
        (pl.col('saldo') != 0).sum().alias('non_zero_rows_saldo'),
        (pl.col('variacion') != 0).sum().alias('non_zero_rows_variacion'),
    ])
    .sort(['grupo', 'nivel_1'])
)
base_summary


## 6. Transformaciones auxiliares para dificultad curricular y global model

Se deja de usar `signed-log` como transformación principal.

Para un modelo global usamos una escala comparable por serie:

- `saldo_model = (saldo - mediana_saldo_serie) / escala_robusta_saldo`
- `variacion_model = variacion / escala_robusta_variacion`

La escala robusta usa IQR y cae a media absoluta si el IQR es cero. Los valores originales se conservan intactos.


In [ ]:
# Robust scaling por cuenta-divisa para global model.
# Nota: las salidas compatibles siguen en escala original; estas columnas son para
# scoring, diagnóstico y entrenamiento global si decides usar escala robusta directa.

robust_stats = (
    base_raw
    .group_by('cross_key_id')
    .agg([
        pl.col('saldo').median().alias('saldo_center'),
        (
            pl.col('saldo').quantile(0.75) -
            pl.col('saldo').quantile(0.25)
        ).alias('saldo_iqr'),
        pl.col('saldo').abs().mean().alias('saldo_abs_mean'),

        # En variación, el cero tiene significado fuerte. Por eso no centramos
        # la serie al transformar; sólo usamos escala robusta de variación.
        pl.col('variacion').median().alias('variacion_center_diagnostic'),
        (
            pl.col('variacion').quantile(0.75) -
            pl.col('variacion').quantile(0.25)
        ).alias('variacion_iqr'),
        pl.col('variacion').abs().mean().alias('variacion_abs_mean'),
    ])
    .with_columns([
        pl.when(pl.col('saldo_iqr').is_not_null() & (pl.col('saldo_iqr').abs() > 0))
        .then(pl.col('saldo_iqr').abs())
        .when(pl.col('saldo_abs_mean').is_not_null() & (pl.col('saldo_abs_mean').abs() > 0))
        .then(pl.col('saldo_abs_mean').abs())
        .otherwise(1.0)
        .alias('saldo_scale'),

        pl.when(pl.col('variacion_iqr').is_not_null() & (pl.col('variacion_iqr').abs() > 0))
        .then(pl.col('variacion_iqr').abs())
        .when(pl.col('variacion_abs_mean').is_not_null() & (pl.col('variacion_abs_mean').abs() > 0))
        .then(pl.col('variacion_abs_mean').abs())
        .otherwise(1.0)
        .alias('variacion_scale'),
    ])
)

base = (
    base_raw
    .join(
        robust_stats.select([
            'cross_key_id',
            'saldo_center',
            'saldo_scale',
            'variacion_center_diagnostic',
            'variacion_scale',
        ]),
        on='cross_key_id',
        how='left',
    )
    .with_columns([
        ((pl.col('saldo') - pl.col('saldo_center')) / pl.col('saldo_scale')).alias('saldo_model'),
        (pl.col('variacion') / pl.col('variacion_scale')).alias('variacion_model'),
        (pl.col('saldo') != 0).cast(pl.Int8).alias('saldo_non_zero'),
        (pl.col('variacion') != 0).cast(pl.Int8).alias('variacion_non_zero'),
    ])
)

# Clipping sólo para métricas de dificultad/visualización; no sustituye los valores originales.
base = base.with_columns([
    pl.when(pl.col('saldo_model') < -20).then(-20.0)
    .when(pl.col('saldo_model') > 20).then(20.0)
    .otherwise(pl.col('saldo_model'))
    .alias('saldo_model_clip'),

    pl.when(pl.col('variacion_model') < -20).then(-20.0)
    .when(pl.col('variacion_model') > 20).then(20.0)
    .otherwise(pl.col('variacion_model'))
    .alias('variacion_model_clip'),
])

base.select([
    'cross_key_id',
    'cutoff_date',
    'saldo',
    'saldo_model',
    'variacion',
    'variacion_model',
    'saldo_center',
    'saldo_scale',
    'variacion_scale',
]).head(10)


In [ ]:
def get_unique_quantile_edges(df: pl.DataFrame, col: str, n_bins: int) -> list[float]:
    probs = [i / n_bins for i in range(1, n_bins)]
    values = []
    for p in probs:
        q = df.select(pl.col(col).quantile(p)).item()
        if q is not None and math.isfinite(float(q)):
            values.append(float(q))
    return sorted(set(values))


def bin_expr(col: str, edges: list[float], output_col: str) -> pl.Expr:
    expr = pl.lit(0)
    for i, edge in enumerate(edges, start=1):
        expr = pl.when(pl.col(col) > edge).then(i).otherwise(expr)
    return expr.cast(pl.Int16).alias(output_col)

saldo_edges = get_unique_quantile_edges(base, 'saldo_model_clip', N_BINS_ENTROPY)
variacion_edges = get_unique_quantile_edges(base, 'variacion_model_clip', N_BINS_ENTROPY)

print('saldo_edges:', saldo_edges)
print('variacion_edges:', variacion_edges)

base = base.with_columns([
    bin_expr('saldo_model_clip', saldo_edges, 'saldo_state'),
    bin_expr('variacion_model_clip', variacion_edges, 'variacion_state'),
])

base.select([
    'saldo_model_clip',
    'saldo_state',
    'variacion_model_clip',
    'variacion_state',
]).head(10)


## 7. Métricas curriculares en Polars

La dificultad combina métricas normalizadas sobre la escala robusta:

- Entropía del saldo robust-scaled.
- Entropía de la variación robust-scaled.
- Varianza del saldo robust-scaled.
- Varianza de la variación robust-scaled.


In [ ]:
def entropy_by_state(df: pl.DataFrame, state_col: str, output_col: str) -> pl.DataFrame:
    counts = (
        df
        .group_by(['cross_key_id', state_col])
        .agg(pl.len().alias('state_count'))
    )

    totals = (
        df
        .group_by('cross_key_id')
        .agg(pl.len().alias('n_obs_entropy'))
    )

    return (
        counts
        .join(totals, on='cross_key_id', how='left')
        .with_columns((pl.col('state_count') / pl.col('n_obs_entropy')).alias('p'))
        .with_columns((-pl.col('p') * pl.col('p').log()).alias('entropy_component'))
        .group_by('cross_key_id')
        .agg((pl.col('entropy_component').sum() / math.log(N_BINS_ENTROPY)).alias(output_col))
    )

entropy_saldo = entropy_by_state(base, 'saldo_state', 'entropy_saldo')
entropy_variacion = entropy_by_state(base, 'variacion_state', 'entropy_variacion')

stats = (
    base
    .group_by([
        'cross_key_id',
        'cuenta',
        'divisa',
        'grupo',
        'nombre_grupo',
        'nivel_1',
        'classification_account_type',
        'gl_account_hist_behaviour_type',
        'currency_id',
    ])
    .agg([
        pl.len().alias('n_obs'),
        pl.col('n_llaves').max().alias('n_llaves'),
        pl.col('n_interfaces').max().alias('n_interfaces'),
        pl.col('n_centros').max().alias('n_centros'),
        pl.col('n_llaves_cuadre').max().alias('n_llaves_cuadre'),
        pl.col('n_llaves_descuadre').max().alias('n_llaves_descuadre'),
        pl.col('total_diferencia_estanco').max().alias('total_diferencia_estanco'),
        pl.col('saldo_center').max().alias('saldo_center'),
        pl.col('saldo_scale').max().alias('saldo_scale'),
        pl.col('variacion_scale').max().alias('variacion_scale'),
        pl.col('saldo_non_zero').sum().alias('non_zero_days_saldo'),
        pl.col('variacion_non_zero').sum().alias('non_zero_days_variacion'),
        pl.col('saldo').abs().mean().alias('avg_abs_balance'),
        pl.col('saldo_model_clip').var().fill_null(0).alias('var_saldo_model'),
        pl.col('variacion_model_clip').var().fill_null(0).alias('var_variacion_model'),
    ])
    .join(entropy_saldo, on='cross_key_id', how='left')
    .join(entropy_variacion, on='cross_key_id', how='left')
    .with_columns([
        pl.col('entropy_saldo').fill_null(0),
        pl.col('entropy_variacion').fill_null(0),
        pl.col('var_saldo_model').fill_null(0),
        pl.col('var_variacion_model').fill_null(0),
    ])
)

stats.head(10)


In [ ]:
def minmax_expr(col: str, output_col: str) -> pl.Expr:
    min_v = pl.col(col).min()
    max_v = pl.col(col).max()
    return (
        pl.when((max_v - min_v) == 0)
        .then(0.0)
        .otherwise((pl.col(col) - min_v) / (max_v - min_v))
        .alias(output_col)
    )

stats = (
    stats
    .with_columns([
        minmax_expr('entropy_saldo', 'entropy_saldo_norm'),
        minmax_expr('entropy_variacion', 'entropy_variacion_norm'),
        minmax_expr('var_saldo_model', 'var_saldo_model_norm'),
        minmax_expr('var_variacion_model', 'var_variacion_model_norm'),
    ])
    .with_columns(
        (
            0.35 * pl.col('entropy_saldo_norm') +
            0.25 * pl.col('entropy_variacion_norm') +
            0.20 * pl.col('var_saldo_model_norm') +
            0.20 * pl.col('var_variacion_model_norm')
        ).alias('difficulty_score')
    )
)

# Buckets curriculares globales de baja a alta dificultad.
stats = (
    stats
    .sort(['difficulty_score', 'cross_key_id'])
    .with_row_index('global_rank')
    .with_columns(
        (
            ((pl.col('global_rank') * N_CURRICULUM_BUCKETS) / pl.len()).floor() + 1
        )
        .clip(1, N_CURRICULUM_BUCKETS)
        .cast(pl.Int16)
        .alias('curriculum_bucket')
    )
)

stats.select([
    'cross_key_id',
    'grupo',
    'nivel_1',
    'gl_account_hist_behaviour_type',
    'difficulty_score',
    'curriculum_bucket',
    'entropy_saldo',
    'entropy_variacion',
    'var_saldo_model',
    'var_variacion_model',
]).head(20)


## 8. Selección curricular sobre Universo 3 AS-OF

La base de Universo 3 se construye como:

```text
Universo 1 histórico AS-OF 2026-04-30
∩
Universo 2 DETA Nivel 1 CUADRE/DESCUADRE
∩
Grupos entrenables: Grupo_2, Grupo_3, Grupo_5
```

Para el modelado final se conservan **sólo las cuenta-divisa con `difficulty_score >= MODEL_SCORE_THRESHOLD`**.

Con `MODEL_SCORE_THRESHOLD = 0.5`, `full_diff.csv` y `full_diff_robust.csv` contienen únicamente esas series modelables.

`SAMPLE_MODE=True` queda sólo como interruptor de emergencia para smoke tests rápidos; para este análisis debe permanecer en `False`.


In [ ]:
# Universo modelable LEGACY para code_02: sólo series arriba del umbral de dificultad.
# Este filtro NO se aplica al dataset global de Financial-GFM.
stats_modelable = (
    stats
    .filter(pl.col('difficulty_score') >= MODEL_SCORE_THRESHOLD)
    .sort(['grupo', 'nivel_1', 'difficulty_score', 'cross_key_id'], descending=[False, False, True, False])
)

print('Universo 3 AS-OF total cuenta-divisa (fuente global):', stats.get_column('cross_key_id').n_unique())
print(f'Cuenta-divisa seleccionadas sólo para modelos locales con difficulty_score >= {MODEL_SCORE_THRESHOLD}:', stats_modelable.get_column('cross_key_id').n_unique())

threshold_validation = (
    stats_modelable
    .group_by(['grupo', 'nombre_grupo', 'nivel_1'])
    .agg([
        pl.col('cross_key_id').n_unique().alias('n_cuenta_divisa_modelables'),
        pl.col('difficulty_score').min().alias('min_difficulty'),
        pl.col('difficulty_score').max().alias('max_difficulty'),
        pl.col('curriculum_bucket').min().alias('min_curriculum_bucket'),
        pl.col('curriculum_bucket').max().alias('max_curriculum_bucket'),
    ])
    .sort(['grupo', 'nivel_1'])
)

display(threshold_validation)

if stats_modelable.height == 0:
    raise ValueError(
        f'No hay cuenta-divisa con difficulty_score >= {MODEL_SCORE_THRESHOLD}. '
        'Baja MODEL_SCORE_THRESHOLD o revisa el cálculo de difficulty_score.'
    )

if SAMPLE_MODE:
    stats_sorted = (
        stats_modelable
        .sort(['grupo', 'nivel_1', 'difficulty_score', 'cross_key_id'], descending=[False, False, True, False])
        .with_columns([
            pl.int_range(0, pl.len()).over(['grupo', 'nivel_1']).alias('rank_in_group'),
            pl.len().over(['grupo', 'nivel_1']).alias('n_group'),
        ])
        .with_columns(
            (
                (pl.col('rank_in_group') * N_PER_GROUP / pl.col('n_group'))
                .floor()
                .cast(pl.Int16)
                .clip(0, N_PER_GROUP - 1)
            ).alias('selection_slot')
        )
    )

    selected_curriculum_table = (
        stats_sorted
        .group_by(['grupo', 'nivel_1', 'selection_slot'])
        .agg([
            pl.col('cross_key_id').first(),
            pl.col('cuenta').first(),
            pl.col('divisa').first(),
            pl.col('nombre_grupo').first(),
            pl.col('classification_account_type').first(),
            pl.col('gl_account_hist_behaviour_type').first(),
            pl.col('currency_id').first(),
            pl.col('n_obs').first(),
            pl.col('n_llaves').first(),
            pl.col('n_interfaces').first(),
            pl.col('n_centros').first(),
            pl.col('n_llaves_cuadre').first(),
            pl.col('n_llaves_descuadre').first(),
            pl.col('total_diferencia_estanco').first(),
            pl.col('non_zero_days_saldo').first(),
            pl.col('non_zero_days_variacion').first(),
            pl.col('avg_abs_balance').first(),
            pl.col('saldo_center').first(),
            pl.col('saldo_scale').first(),
            pl.col('variacion_scale').first(),
            pl.col('entropy_saldo').first(),
            pl.col('entropy_variacion').first(),
            pl.col('var_saldo_model').first(),
            pl.col('var_variacion_model').first(),
            pl.col('entropy_saldo_norm').first(),
            pl.col('entropy_variacion_norm').first(),
            pl.col('var_saldo_model_norm').first(),
            pl.col('var_variacion_model_norm').first(),
            pl.col('difficulty_score').first(),
            pl.col('curriculum_bucket').first(),
        ])
        .with_columns((pl.col('selection_slot') + 1).alias('selection_slot'))
        .sort(['grupo', 'nivel_1', 'selection_slot'])
    )
else:
    selected_curriculum_table = (
        stats_modelable
        .with_columns(pl.lit(None).cast(pl.Int16).alias('selection_slot'))
        .sort(['grupo', 'nivel_1', 'difficulty_score', 'cross_key_id'], descending=[False, False, True, False])
    )

selected_curriculum_table


In [ ]:
selection_validation = (
    selected_curriculum_table
    .group_by(['grupo', 'nombre_grupo', 'nivel_1'])
    .agg([
        pl.col('cross_key_id').n_unique().alias('selected_cuenta_divisa'),
        pl.col('curriculum_bucket').min().alias('min_curriculum_bucket'),
        pl.col('curriculum_bucket').max().alias('max_curriculum_bucket'),
        pl.col('difficulty_score').min().alias('min_difficulty'),
        pl.col('difficulty_score').max().alias('max_difficulty'),
    ])
    .sort(['grupo', 'nivel_1'])
)

print(f'Selección legacy para code_02: difficulty_score >= {MODEL_SCORE_THRESHOLD}')
selection_validation


## 9. Base seleccionada y datasets largos

Aquí se usan **todas** las cuenta-divisa seleccionadas cuando `SAMPLE_MODE=False`.

Se generan dos versiones:

1. `series_long`: valores originales, compatible con el flujo anterior.
2. `series_long_robust`: valores robust-scaled, útil para experimentar con un global model directo.


In [ ]:
selected_ids = selected_curriculum_table.get_column('cross_key_id').to_list()
print('Cuenta-divisa seleccionadas:', len(selected_ids))

base_selected = (
    base
    .filter(pl.col('cross_key_id').is_in(selected_ids))
    .join(
        selected_curriculum_table.select([
            'cross_key_id',
            'curriculum_bucket',
            'selection_slot',
            'difficulty_score',
        ]),
        on='cross_key_id',
        how='left',
    )
)

print(base_selected.shape)
base_selected.head(10)


In [ ]:
# Formato largo enriquecido para auditoría en escala original.
series_long_saldo = (
    base_selected
    .select([
        pl.col('cutoff_date').alias('fecha'),
        pl.col('cross_key_id'),
        pl.col('cuenta'),
        pl.col('divisa'),
        pl.col('currency_id'),
        pl.col('grupo'),
        pl.col('nombre_grupo'),
        pl.col('nivel_1'),
        pl.col('classification_account_type'),
        pl.col('gl_account_hist_behaviour_type'),
        pl.col('universe_label_cutoff_date'),
        pl.col('raw_hist_classification_account_type'),
        pl.col('raw_hist_behaviour_type'),
        pl.col('curriculum_bucket'),
        pl.col('selection_slot'),
        pl.col('difficulty_score'),
        pl.col('n_llaves'),
        pl.col('n_interfaces'),
        pl.col('n_centros'),
        pl.col('n_llaves_cuadre'),
        pl.col('n_llaves_descuadre'),
        pl.col('total_diferencia_estanco'),
        pl.col('saldo_center'),
        pl.col('saldo_scale'),
        pl.col('variacion_scale'),
        pl.lit('saldo').alias('tipo_serie'),
        (pl.col('cross_key_id') + pl.lit('_saldo')).alias('serie'),
        pl.col('saldo').alias('total_amount'),
        pl.col('saldo_model').alias('total_amount_robust'),
    ])
)

series_long_variacion = (
    base_selected
    .select([
        pl.col('cutoff_date').alias('fecha'),
        pl.col('cross_key_id'),
        pl.col('cuenta'),
        pl.col('divisa'),
        pl.col('currency_id'),
        pl.col('grupo'),
        pl.col('nombre_grupo'),
        pl.col('nivel_1'),
        pl.col('classification_account_type'),
        pl.col('gl_account_hist_behaviour_type'),
        pl.col('universe_label_cutoff_date'),
        pl.col('raw_hist_classification_account_type'),
        pl.col('raw_hist_behaviour_type'),
        pl.col('curriculum_bucket'),
        pl.col('selection_slot'),
        pl.col('difficulty_score'),
        pl.col('n_llaves'),
        pl.col('n_interfaces'),
        pl.col('n_centros'),
        pl.col('n_llaves_cuadre'),
        pl.col('n_llaves_descuadre'),
        pl.col('total_diferencia_estanco'),
        pl.col('saldo_center'),
        pl.col('saldo_scale'),
        pl.col('variacion_scale'),
        pl.lit('variacion').alias('tipo_serie'),
        (pl.col('cross_key_id') + pl.lit('_variacion')).alias('serie'),
        pl.col('variacion').alias('total_amount'),
        pl.col('variacion_model').alias('total_amount_robust'),
    ])
)

series_long = (
    pl.concat([series_long_saldo, series_long_variacion])
    .sort(['fecha', 'serie'])
)

series_long_robust = (
    series_long
    .drop('total_amount')
    .rename({'total_amount_robust': 'total_amount'})
    .sort(['fecha', 'serie'])
)


# Dataset canónico de Financial-GFM (Checkpoint 19).
# IMPORTANTE:
# - usa TODO Universo 3 elegible, no sólo difficulty_score >= MODEL_SCORE_THRESHOLD;
# - usa únicamente targets en escala original;
# - difficulty_score/nivel_curriculum son placeholders de esquema;
# - GlobalDatasetFactory crea primero el split y después recalcula dificultad
#   sólo con train y por cross_key_id final (saldo/variación separados).
global_series_long_saldo_source = (
    base
    .select([
        pl.col('cutoff_date').alias('fecha'),
        pl.col('cross_key_id'),
        pl.col('divisa'),
        pl.lit('saldo').alias('tipo_serie'),
        pl.col('saldo').alias('total_amount'),
        pl.lit(0.0).cast(pl.Float64).alias('difficulty_score'),
        pl.lit(1).cast(pl.Int16).alias('curriculum_bucket'),
        pl.col('grupo'),
    ])
)

global_series_long_variacion_source = (
    base
    .select([
        pl.col('cutoff_date').alias('fecha'),
        pl.col('cross_key_id'),
        pl.col('divisa'),
        pl.lit('variacion').alias('tipo_serie'),
        pl.col('variacion').alias('total_amount'),
        pl.lit(0.0).cast(pl.Float64).alias('difficulty_score'),
        pl.lit(1).cast(pl.Int16).alias('curriculum_bucket'),
        pl.col('grupo'),
    ])
)

global_series_long_source = (
    pl.concat([
        global_series_long_saldo_source,
        global_series_long_variacion_source,
    ])
    .sort(['cross_key_id', 'tipo_serie', 'fecha'])
)

global_series_long, global_long_report = build_and_validate_global_long(
    global_series_long_source
)

print('global_series_long canónico:', global_series_long.shape)
print('global_long_report:', global_long_report.to_dict())
print(
    'Cuenta-divisa globales / series finales:',
    global_series_long.get_column('account_currency_id').n_unique(),
    '/',
    global_series_long.get_column('cross_key_id').n_unique(),
)
print('series_long local original:', series_long.shape)
print('series_long local robust:', series_long_robust.shape)
global_series_long.head(20)


## 10. Formato original del notebook anterior: `series_insumo`

Se conserva el formato:

```text
load_date | serie | total_amount
```

- `series_insumo`: escala original.
- `series_insumo_robust`: escala robusta por serie, opcional para global model.


In [ ]:
series_insumo = (
    series_long
    .select([
        pl.col('fecha').alias('load_date'),
        pl.col('serie'),
        pl.col('total_amount'),
    ])
    .sort(['load_date', 'serie'])
)

series_insumo_robust = (
    series_long_robust
    .select([
        pl.col('fecha').alias('load_date'),
        pl.col('serie'),
        pl.col('total_amount'),
    ])
    .sort(['load_date', 'serie'])
)

print('series_insumo original:', series_insumo.shape)
print('series_insumo_robust:', series_insumo_robust.shape)
series_insumo.head(30)


In [ ]:
try:
    series_insumo_pivot = (
        series_insumo
        .pivot(
            index='load_date',
            on='serie',
            values='total_amount',
            aggregate_function='first',
        )
        .sort('load_date')
    )
    series_insumo_robust_pivot = (
        series_insumo_robust
        .pivot(
            index='load_date',
            on='serie',
            values='total_amount',
            aggregate_function='first',
        )
        .sort('load_date')
    )
except TypeError:
    # Compatibilidad con Polars anterior.
    series_insumo_pivot = (
        series_insumo
        .pivot(
            index='load_date',
            columns='serie',
            values='total_amount',
            aggregate_function='first',
        )
        .sort('load_date')
    )
    series_insumo_robust_pivot = (
        series_insumo_robust
        .pivot(
            index='load_date',
            columns='serie',
            values='total_amount',
            aggregate_function='first',
        )
        .sort('load_date')
    )

print('series_insumo_pivot original:', series_insumo_pivot.shape)
print('series_insumo_robust_pivot:', series_insumo_robust_pivot.shape)
series_insumo_pivot.head(5)


## 11. Formato compatible con `code_02_MLP_E_D.ipynb`: `full_diff.csv`

`code_02` espera un CSV wide con columna `fecha`. Por eso tomamos `series_insumo_pivot` y renombramos `load_date` a `fecha`.

- `full_diff`: valores originales.
- `full_diff_robust`: valores robust-scaled por serie, opcional para un global model directo.


In [ ]:
full_diff = series_insumo_pivot.rename({'load_date': 'fecha'})
full_diff_robust = series_insumo_robust_pivot.rename({'load_date': 'fecha'})

expected_n_series = selected_curriculum_table.get_column('cross_key_id').n_unique() * 2
actual_n_series = full_diff.width - 1
actual_n_series_robust = full_diff_robust.width - 1

print('Cuenta-divisa:', selected_curriculum_table.get_column('cross_key_id').n_unique())
print('Series esperadas:', expected_n_series)
print('Series reales original:', actual_n_series)
print('Series reales robust:', actual_n_series_robust)
print('Fechas:', full_diff.height)

assert 'fecha' in full_diff.columns
assert 'fecha' in full_diff_robust.columns
assert actual_n_series == expected_n_series, f'Original: esperadas={expected_n_series}, reales={actual_n_series}'
assert actual_n_series_robust == expected_n_series, f'Robust: esperadas={expected_n_series}, reales={actual_n_series_robust}'

if SAMPLE_MODE:
    validation_total = selection_validation.get_column('selected_cuenta_divisa').sum()
    assert selected_curriculum_table.get_column('cross_key_id').n_unique() == validation_total

full_diff.head(5)


In [ ]:
null_summary = (
    full_diff
    .null_count()
    .transpose(include_header=True, header_name='column', column_names=['null_count'])
    .sort('null_count', descending=True)
)

null_summary_robust = (
    full_diff_robust
    .null_count()
    .transpose(include_header=True, header_name='column', column_names=['null_count'])
    .sort('null_count', descending=True)
)

print('Nulls original:')
display(null_summary.head(30))
print('Nulls robust:')
display(null_summary_robust.head(30))


## 12. Gráficas diagnósticas del currículo

Las gráficas ya no comparan contra `signed-log`. Ahora se muestran:

- Serie original.
- Serie robust-scaled.

No se mezclan en el mismo eje para evitar lecturas raras.


In [ ]:
# 1. Escalera de dificultad por bucket del universo modelable.
bucket_pd = (
    selected_curriculum_table
    .group_by('curriculum_bucket')
    .agg(pl.col('difficulty_score').mean().alias('mean_difficulty'))
    .sort('curriculum_bucket')
    .to_pandas()
)

plt.figure(figsize=(10, 5))
plt.plot(bucket_pd['curriculum_bucket'], bucket_pd['mean_difficulty'], marker='o')
plt.title(f'Escalera curricular modelable: dificultad media por bucket | score >= {MODEL_SCORE_THRESHOLD}')
plt.xlabel('Curriculum bucket')
plt.ylabel('Dificultad media')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 2. Entropía vs varianza de las series modelables en escala robusta.
selected_pd = selected_curriculum_table.to_pandas()

plt.figure(figsize=(8, 6))
for label in selected_pd['nombre_grupo'].unique():
    part = selected_pd[selected_pd['nombre_grupo'] == label]
    plt.scatter(part['entropy_saldo'], part['var_saldo_model'], label=label)
plt.title(f'Universo 3 modelable: entropía vs varianza robusta del saldo | score >= {MODEL_SCORE_THRESHOLD}')
plt.xlabel('Entropía saldo')
plt.ylabel('Varianza robust-scaled saldo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 3. Selección modelable a lo largo del currículo.
plt.figure(figsize=(10, 5))
for label in selected_pd['nivel_1'].unique():
    part = selected_pd[selected_pd['nivel_1'] == label]
    plt.scatter(part['curriculum_bucket'], part['difficulty_score'], label=label)
plt.axhline(MODEL_SCORE_THRESHOLD, linestyle='--', linewidth=1, alpha=0.5)
plt.title('Universo 3 modelable: dificultad por bucket curricular y Nivel 1')
plt.xlabel('Curriculum bucket')
plt.ylabel('Difficulty score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 4. Diagnóstico visual dirigido.
# Ojo: esto NO muestrea el dataset de entrenamiento; sólo limita las gráficas.
# El dataset de modelado ya quedó filtrado por MODEL_SCORE_THRESHOLD.
# Aquí tomamos hasta 10 CUADRE + 10 DESCUADRE entre las series modelables.

N_PER_STATUS_PLOT = 10

eligible_examples = (
    selected_curriculum_table
    .filter(pl.col('nivel_1').is_in(['CUADRE', 'DESCUADRE']))
    .filter(pl.col('difficulty_score') >= MODEL_SCORE_THRESHOLD)
    .sort(['nivel_1', 'difficulty_score', 'grupo', 'cross_key_id'], descending=[False, True, False, False])
    .select([
        'cross_key_id',
        'grupo',
        'nombre_grupo',
        'nivel_1',
        'curriculum_bucket',
        'selection_slot',
        'difficulty_score',
        'entropy_saldo',
        'var_saldo_model',
    ])
)

print(
    eligible_examples
    .group_by('nivel_1')
    .agg([
        pl.len().alias('n_series_disponibles'),
        pl.col('difficulty_score').min().alias('min_difficulty'),
        pl.col('difficulty_score').max().alias('max_difficulty'),
    ])
    .sort('nivel_1')
)

examples_20 = pl.concat([
    eligible_examples
    .filter(pl.col('nivel_1') == 'CUADRE')
    .sort('difficulty_score', descending=True)
    .head(N_PER_STATUS_PLOT),

    eligible_examples
    .filter(pl.col('nivel_1') == 'DESCUADRE')
    .sort('difficulty_score', descending=True)
    .head(N_PER_STATUS_PLOT),
]).sort(['nivel_1', 'difficulty_score'], descending=[False, True])

example_ids = examples_20.get_column('cross_key_id').to_list()
N_PLOT_EXAMPLES = len(example_ids)

print(f'Series seleccionadas para graficar: {N_PLOT_EXAMPLES}')
print(
    examples_20
    .group_by('nivel_1')
    .agg(pl.len().alias('n_plot'))
    .sort('nivel_1')
)

plot_df = (
    base_selected
    .filter(pl.col('cross_key_id').is_in(example_ids))
    .select([
        'cross_key_id',
        'cutoff_date',
        'saldo',
        'variacion',
        'saldo_model',
        'variacion_model',
    ])
    .sort(['cross_key_id', 'cutoff_date'])
    .to_pandas()
)

for idx, row in enumerate(examples_20.iter_rows(named=True), start=1):
    cross_key = row['cross_key_id']
    grupo = row['grupo']
    nombre_grupo = row['nombre_grupo']
    nivel_1 = row['nivel_1']
    curriculum_bucket = row['curriculum_bucket']
    selection_slot = row['selection_slot']
    difficulty_score = row['difficulty_score']
    entropy_saldo = row['entropy_saldo']
    var_saldo_model = row['var_saldo_model']

    part = plot_df[plot_df['cross_key_id'] == cross_key].copy()

    title_base = (
        f'{idx:02d}/{N_PLOT_EXAMPLES} | {cross_key} | '
        f'{grupo} | {nivel_1} | {nombre_grupo} | '
        f'curriculum={curriculum_bucket} | slot={selection_slot} | '
        f'difficulty={difficulty_score:.4f} | '
        f'entropy={entropy_saldo:.4f} | var_robust={var_saldo_model:.4f}'
    )

    # ------------------------------
    # 4.1 Saldo: original vs robust-scaled
    # ------------------------------
    fig, ax1 = plt.subplots(figsize=(12, 4))

    ax1.plot(
        part['cutoff_date'],
        part['saldo'],
        label='saldo original',
    )
    ax1.set_xlabel('cutoff_date')
    ax1.set_ylabel('saldo original')
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(
        part['cutoff_date'],
        part['saldo_model'],
        color='red',
        linestyle='--',
        label='saldo robust-scaled',
    )
    ax2.set_ylabel('saldo robust-scaled')

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

    plt.title(f'Saldo | {title_base}')
    plt.show()

    # ------------------------------
    # 4.2 Variación: original vs robust-scaled
    # ------------------------------
    fig, ax1 = plt.subplots(figsize=(12, 4))

    ax1.plot(
        part['cutoff_date'],
        part['variacion'],
        label='variación original',
    )
    ax1.set_xlabel('cutoff_date')
    ax1.set_ylabel('variación original')
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(
        part['cutoff_date'],
        part['variacion_model'],
        color='red',
        linestyle='--',
        label='variación robust-scaled',
    )
    ax2.set_ylabel('variación robust-scaled')

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

    plt.title(f'Variación | {title_base}')
    plt.show()


## 13. Guardar CSV y Parquet

Por seguridad, `WRITE_OUTPUTS = False` por defecto.

Cuando lo cambies a `True`, se guardarán:

### Compatible con `code_02`
- `full_diff.csv` en escala original.
- `full_diff_robust.csv` en escala robusta por serie.

### Dataset canónico del modelo global
- `global_series_long.csv`
- `global_series_long.parquet`

### Auditoría/diagnóstico legacy
- `series_long.csv`
- `series_long_robust.csv`
- `selected_curriculum_table.csv`
- `curriculum_stats_all_series.csv`

### Respaldo técnico
- Parquets equivalentes bajo `PATH_PARQUET`.


`global_series_long` usa exactamente: `fecha`, `account_currency_id`, `cross_key_id`, `tipo_serie`, `target`, `difficulty_score`, `nivel_curriculum`, `grupo`. No usa la versión robust global; la escala se calculará sólo con la ventana histórica en Checkpoint 2.


In [ ]:
WRITE_OUTPUTS = True

if WRITE_OUTPUTS:
    # ------------------------------
    # CSV compatibles con code_02 / global model
    # ------------------------------
    write_polars_csv_to_s3(full_diff, PATH_FULL_DIFF_CSV)
    write_polars_csv_to_s3(full_diff_robust, PATH_FULL_DIFF_ROBUST_CSV)
    write_polars_csv_to_s3(series_long, PATH_SERIES_LONG_CSV)
    write_polars_csv_to_s3(series_long_robust, PATH_SERIES_LONG_ROBUST_CSV)
    write_polars_csv_to_s3(global_series_long, PATH_GLOBAL_SERIES_LONG_CSV)
    write_polars_parquet_to_s3(global_series_long, PATH_GLOBAL_SERIES_LONG_PARQUET)
    write_text_to_s3(
        json.dumps(global_long_report.to_dict(), ensure_ascii=False, indent=2),
        PATH_GLOBAL_LONG_VALIDATION_JSON,
        content_type='application/json',
    )
    write_polars_csv_to_s3(selected_curriculum_table, PATH_SELECTED_CURRICULUM_CSV)
    write_polars_csv_to_s3(stats, PATH_CURRICULUM_STATS_CSV)

    # ------------------------------
    # CSV técnicos del experimento
    # ------------------------------
    write_polars_csv_to_s3(series_insumo, f'{PATH_CSV}series_insumo.csv')
    write_polars_csv_to_s3(series_insumo_robust, f'{PATH_CSV}series_insumo_robust.csv')
    write_polars_csv_to_s3(series_insumo_pivot, f'{PATH_CSV}series_insumo_pivot.csv')
    write_polars_csv_to_s3(series_insumo_robust_pivot, f'{PATH_CSV}series_insumo_robust_pivot.csv')
    write_polars_csv_to_s3(full_diff, f'{PATH_CSV}full_diff.csv')
    write_polars_csv_to_s3(full_diff_robust, f'{PATH_CSV}full_diff_robust.csv')
    write_polars_csv_to_s3(selected_curriculum_table, f'{PATH_CSV}selected_curriculum_table.csv')
    write_polars_csv_to_s3(stats, f'{PATH_CSV}curriculum_stats_all_series.csv')
    write_polars_csv_to_s3(global_series_long, f'{PATH_CSV}global_series_long.csv')

    # ------------------------------
    # Parquet técnicos del experimento
    # ------------------------------
    write_polars_parquet_to_s3(series_insumo, f'{PATH_PARQUET}series_insumo.parquet')
    write_polars_parquet_to_s3(series_insumo_robust, f'{PATH_PARQUET}series_insumo_robust.parquet')
    write_polars_parquet_to_s3(series_insumo_pivot, f'{PATH_PARQUET}series_insumo_pivot.parquet')
    write_polars_parquet_to_s3(series_insumo_robust_pivot, f'{PATH_PARQUET}series_insumo_robust_pivot.parquet')
    write_polars_parquet_to_s3(full_diff, f'{PATH_PARQUET}full_diff.parquet')
    write_polars_parquet_to_s3(full_diff_robust, f'{PATH_PARQUET}full_diff_robust.parquet')
    write_polars_parquet_to_s3(series_long, f'{PATH_PARQUET}series_long.parquet')
    write_polars_parquet_to_s3(series_long_robust, f'{PATH_PARQUET}series_long_robust.parquet')
    write_polars_parquet_to_s3(selected_curriculum_table, f'{PATH_PARQUET}selected_curriculum_table.parquet')
    write_polars_parquet_to_s3(stats, f'{PATH_PARQUET}curriculum_stats_all_series.parquet')
    write_polars_parquet_to_s3(global_series_long, f'{PATH_PARQUET}global_series_long.parquet')

    print('Outputs escritos correctamente.')
    print('Archivo principal compatible con code_02, escala original:')
    print(PATH_FULL_DIFF_CSV)
    print('Archivo robust legacy/diagnóstico:')
    print(PATH_FULL_DIFF_ROBUST_CSV)
    print('Dataset canónico del modelo global:')
    print(PATH_GLOBAL_SERIES_LONG_PARQUET)
else:
    print('WRITE_OUTPUTS=False. No se escribió ningún archivo.')
    print('Cuando estés listo, cambia WRITE_OUTPUTS=True y ejecuta esta celda.')
    print('Archivo principal compatible con code_02 quedaría en:')
    print(PATH_FULL_DIFF_CSV)
    print('Archivo robust para global model quedaría en:')
    print(PATH_FULL_DIFF_ROBUST_CSV)
    print('Dataset canónico del modelo global:')
    print(PATH_GLOBAL_SERIES_LONG_PARQUET)


## 14. Resumen final esperado

Con `SAMPLE_MODE=False`, este notebook obtiene el **Universo 3 AS-OF completo**:

```text
Grupo_2 / Grupo_3 / Grupo_5
x
CUADRE / DESCUADRE
```

La diferencia contra la versión anterior es importante:

- Antes una `cross_key_id` podía entrar si **alguna vez** tuvo estado entrenable.
- Ahora entra sólo si su **último estado conocido <= FECHA_ANALISIS** es entrenable.

Esto evita títulos contradictorios como:

```text
Grupo_2 | CUADRE | cuentas_atipicos
```

porque `Grupo_2` queda congelado como:

```text
positiva / cuentas_entrenamiento
```

Salidas principales:

- `full_diff.csv`: escala original, compatible con `code_02_MLP_E_D.ipynb`.
- `full_diff_robust.csv`: escala robust-scaled legacy/diagnóstica; no será input del modelo global.
- `series_long.csv` y `series_long_robust.csv`: formato largo legacy para auditoría.
- `global_series_long.parquet`: TODO Universo 3 elegible en escala original; incluye divisa y edad causal. La dificultad guardada aquí es sólo placeholder y se recalcula train-only después del split.
- `global_series_long_validation.json`: evidencia de filas, series, fechas, dificultad y curriculum.
- `selected_curriculum_table.csv`: selección legacy para los modelos locales code_02; no filtra Financial-GFM.

Revisa `selection_validation` y `base_summary` después de correr para obtener los conteos reales AS-OF.


Checkpoint 1 no crea ventanas ni entrena modelos. Los cuatro `code_02_*` continúan usando `full_diff.csv` sin cambios.


## Contrato Checkpoint 19

`code_01` ya no decide la dificultad curricular del GFM. Los cuatro `code_03_*` deben volver a cargar este nuevo `global_series_long.parquet`; su `GlobalDatasetFactory` crea el split por cuenta-divisa y recalcula dificultad sólo con train.
